<a href="https://colab.research.google.com/github/Mmbsaksd/Full_Stack_GenerativeAI_And_Agentic_AI/blob/main/LLM_Fine_Tuning_Non_Instruction%2C_Instruction%2C_and_Preference_Data_with_Huggingface.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Stage 1:** Non-instruction Causal LLM Fine-tuning or Domain-Adaptive Continued Pretraining

In [ ]:
# ============================================================
# 1. Install required libraries
# ============================================================
# PyMuPDF: PDF text extraction
# datasets: Hugging Face dataset creation
# transformers/accelerate: model, tokenizer, Trainer
# peft: LoRA/QLoRA adapters
# bitsandbytes: 4-bit/8-bit quantized loading

!pip install -q -U pymupdf datasets transformers accelerate peft bitsandbytes torchao

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.7/25.7 MB 81.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 555.1/555.1 kB 24.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.5/11.5 MB 96.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 11.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 75.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 14.1 MB/s eta 0:00:00


In [ ]:
import warnings
warnings.filterwarnings("ignore")

In [11]:
# ============================================================
# 3. Global configuration
# ============================================================
# Keep all important parameters in one place.
# This makes the notebook easier to debug, reproduce, and productionize.


from dataclasses import dataclass, asdict

@dataclass
class Config:
    pdf_path: str = "/content/Metformin-Lipid-Therapy-Knowledge (1).pdf"
    model_name: str = "TinyLlama/TinyLlama-1.1B-intermediate-step-1431k-3T"
    output_dir: str = "/content/pharma_tinyllama_lora_output"
    adapter_dir: str = "/content/pharma_tinyllama_lora_adapter"
    processed_data_dir: str = "/content/pharma_processed_data"
    min_chars_per_paragraph: int = 80
    block_size: int = 512
    test_size: float = 0.15
    seed: int = 42
    lora_r: int = 16
    lora_alpha: int = 32
    lora_dropout: float = 0.05
    num_train_epochs: float = 10.0
    per_device_train_batch_size: int = 1
    per_device_eval_batch_size: int = 1
    gradient_accumulation_steps: int = 8
    learning_rate: float = 2e-4
    warmup_ratio: float = 0.03
    weight_decay: float = 0.01
    logging_steps=1
    logging_first_step=True
    eval_steps: int = 10
    save_steps: int = 25
    save_total_limit: int = 2


In [12]:
config = Config()

In [13]:
config

Config(pdf_path='/content/Metformin-Lipid-Therapy-Knowledge (1).pdf', model_name='TinyLlama/TinyLlama-1.1B-intermediate-step-1431k-3T', output_dir='/content/pharma_tinyllama_lora_output', adapter_dir='/content/pharma_tinyllama_lora_adapter', processed_data_dir='/content/pharma_processed_data', min_chars_per_paragraph=80, block_size=512, test_size=0.15, seed=42, lora_r=16, lora_alpha=32, lora_dropout=0.05, num_train_epochs=10.0, per_device_train_batch_size=1, per_device_eval_batch_size=1, gradient_accumulation_steps=8, learning_rate=0.0002, warmup_ratio=0.03, weight_decay=0.01, eval_steps=10, save_steps=25, save_total_limit=2)

In [14]:
import json
print(json.dumps(asdict(config), indent=2))

{
  "pdf_path": "/content/Metformin-Lipid-Therapy-Knowledge (1).pdf",
  "model_name": "TinyLlama/TinyLlama-1.1B-intermediate-step-1431k-3T",
  "output_dir": "/content/pharma_tinyllama_lora_output",
  "adapter_dir": "/content/pharma_tinyllama_lora_adapter",
  "processed_data_dir": "/content/pharma_processed_data",
  "min_chars_per_paragraph": 80,
  "block_size": 512,
  "test_size": 0.15,
  "seed": 42,
  "lora_r": 16,
  "lora_alpha": 32,
  "lora_dropout": 0.05,
  "num_train_epochs": 10.0,
  "per_device_train_batch_size": 1,
  "per_device_eval_batch_size": 1,
  "gradient_accumulation_steps": 8,
  "learning_rate": 0.0002,
  "warmup_ratio": 0.03,
  "weight_decay": 0.01,
  "eval_steps": 10,
  "save_steps": 25,
  "save_total_limit": 2
}


In [15]:
config.output_dir

'/content/pharma_tinyllama_lora_output'

In [16]:
config.processed_data_dir

'/content/pharma_processed_data'

In [17]:
import os
os.makedirs(config.output_dir, exist_ok=True)
os.makedirs(config.adapter_dir, exist_ok=True)
os.makedirs(config.processed_data_dir, exist_ok=True)

In [18]:
# ============================================================
# 4. Optional Colab upload helper
# ============================================================
# Run this cell only if your PDF is not already present at config.pdf_path.
if not os.path.exists(config.pdf_path):
    print(f"PDF not found at: {config.pdf_path}")
else:
    print(f"PDF found: {config.pdf_path}")

PDF found: /content/Metformin-Lipid-Therapy-Knowledge (1).pdf


In [19]:
# # ============================================================
# # 5. Extract text from PDF
# # ============================================================
from typing import List, Dict, Any
import fitz  # PyMuPDF
def extract_pdf_pages(pdf_path: str) -> List[Dict[str, Any]]:
    # Extract page-level text from a PDF.
    pages = []
    with fitz.open(pdf_path) as doc:
        for page_index, page in enumerate(doc, start=1):
            text = page.get_text("text")
            text = text.strip() if text else ""
            if text:
                pages.append({
                    "page": page_index,
                    "text": text,
                    "char_count": len(text),
                })
    return pages


In [20]:
config.pdf_path

'/content/Metformin-Lipid-Therapy-Knowledge (1).pdf'

In [21]:
pdf_pages = extract_pdf_pages(config.pdf_path)

In [22]:
print(f"Total pages with extracted text: {len(pdf_pages)}")
print("Page-level character counts:")
for item in pdf_pages:
    print(f"Page {item['page']}: {item['char_count']} characters")

Total pages with extracted text: 6
Page-level character counts:
Page 1: 2244 characters
Page 2: 2889 characters
Page 3: 2636 characters
Page 4: 2416 characters
Page 5: 2613 characters
Page 6: 2761 characters


In [23]:
print(pdf_pages[0]["text"])

Metformin is one of the most widely prescribed oral antihyperglycemic agents.​
 Its primary mechanism of action involves the activation of AMP-activated protein kinase 
(AMPK), a central metabolic regulator that promotes glucose uptake and fatty acid oxidation 
while inhibiting hepatic gluconeogenesis.​
 Beyond its glycemic control, Metformin has been shown to improve cardiovascular outcomes 
and display anti-inflammatory properties.​
 Recent studies also suggest potential anticancer effects through inhibition of the mTOR 
signaling pathway and suppression of tumor angiogenesis. 
 
Clinical trials have demonstrated that combining Atorvastatin with Ezetimibe results in 
significant reductions in low-density lipoprotein cholesterol (LDL-C) levels compared to 
monotherapy.​
 Ezetimibe acts by inhibiting the Niemann–Pick C1-like 1 (NPC1L1) transporter in the intestinal 
wall, reducing cholesterol absorption, while Atorvastatin inhibits hepatic HMG-CoA reductase, 
suppressing endogenous cho

In [ ]:
# ============================================================
# 6. Text cleaning utilities
# ============================================================
import re
import unicodedata

def clean_pdf_text(text: str) -> str: